# Claude Token & Cost Estimator

**Note:** This tool was originally created by Anthropic but I am trying to expand, improve, and keep it up to date with the latest models and their cost. If you want to contribute, let me know so we can collaborate.

**How this notebook stays current.** The estimator logic lives in a pip-installable package, [`tce-library`](https://github.com/ashfordreyes/tce-library), rather than in these cells. The setup cell below installs it straight from GitHub, and because a fresh Colab runtime is a clean VM, every session pulls the newest commit automatically — no copying cells between notebooks to get the latest version. `tce.status()` tells you which commit you're running.

These cells now hold only *your* input: the prompt, the context, and the knobs. Everything else is in the package.

**Conversation context (section 4).** You can paste the conversation so far alongside your prompt. The Messages API is stateless, so every prior turn is resent and re-billed on every request: a chat that feels like it costs one prompt's worth actually costs the whole transcript's worth, every turn. Section 7 splits your bill into "new prompt" versus "re-sending what you already sent", projects the next N turns (the cost grows quadratically), and shows what prompt caching would save.

-----------------------------

This notebook uses Anthropic's official **`/v1/messages/count_tokens`** endpoint to get an *exact* token count for a prompt against a specific Claude model, before you actually send it for generation. Counting tokens this way is free and doesn't count against your normal message rate limits.

It then estimates $ cost using current published per-token pricing (input cost is exact since it's based on the real count; output cost is an *estimate* since you choose how many output tokens to assume).

**Note on pricing:** the rates live in the package, in [`pricing.py`](https://github.com/ashfordreyes/tce-library/blob/main/src/tcelibrary/pricing.py), and are current as of July 2026 — but Anthropic can change them. If your numbers look off, check https://www.anthropic.com/pricing, then fix the table in the repo so every notebook picks the change up.

**Note on Sonnet 5 pricing:** it's on introductory pricing ($2/$10 per million tokens) through Aug 31, 2026, then standard pricing ($3/$15) from Sept 1, 2026. Options 4 and 5 in the model menu are the same model at those two rates.

## 1. Setup

Installs the estimator from GitHub and connects to the API. Re-run the whole notebook
in a fresh runtime any time you want the latest code.

In [ ]:
# Pulls the newest estimator code from GitHub. A fresh Colab runtime is a clean VM,
# so this clones main every session — you always get the latest, with nothing to copy.
!pip install -q git+https://github.com/ashfordreyes/tce-library@main

In [ ]:
import tcelibrary as tce

client = tce.connect()   # uses the ANTHROPIC_API_KEY Colab secret if you've set one,
                         # otherwise prompts with the input hidden
tce.status()             # which commit you're running, and whether it's behind

## 2. Models and pricing

The rate tables, cache multipliers and context-window fallbacks live in
[`pricing.py`](https://github.com/ashfordreyes/tce-library/blob/main/src/tcelibrary/pricing.py)
in the package — not in this notebook. That is the point of the split: change a rate
in the repo once, and every notebook picks it up on its next fresh runtime instead of
someone re-pasting a dict.

The cell below shows the exact rates this session will bill at. The model menu itself
appears in section 3, where you pick one.

In [ ]:
# The rates this run will actually bill at, straight from the installed package.
# Keys are pricing keys, not model ids: Sonnet 5 appears twice, at its standard
# and introductory rates.
tce.PRICING

## 3. Choose model + enter your prompt

In [ ]:
model = tce.choose_model()   # pass a number, e.g. tce.choose_model("2"), to skip the prompt

In [ ]:
# Option A: type/paste your prompt directly here
prompt_text = """

"""

# Option B: uncomment to load a prompt from a local .txt file instead
# with open("my_prompt.txt", "r") as f:
#     prompt_text = f.read()

print(f"Prompt loaded ({len(prompt_text)} characters).")

Prompt loaded (18889 characters).


## 4. Optional: existing conversation context

If this prompt isn't the *first* message of a conversation, paste everything that
came before it here.

This matters more than it looks. The Messages API is **stateless** — there is no
conversation living on Anthropic's servers that you're topping up. Every prior
turn is **resent in full, and re-billed, on every single request**. So a chat that
feels like it costs one prompt's worth actually costs the whole transcript's worth,
every turn, and grows *quadratically* as it goes.

Leave it blank and the notebook behaves exactly as it did before.

**Paste format:** by default the context is treated as one opaque blob, which is
accurate to within a handful of tokens. If your transcript uses `User:` /
`Assistant:` turn markers and you want the exact per-message accounting the API
actually bills, set `PARSE_CONTEXT_TURNS = True`.

In [ ]:
# Option A: paste the conversation so far here. Leave blank for a fresh prompt.
context_text = """

"""

# Option B: uncomment to load the context from a local .txt file instead
# with open("my_context.txt", "r") as f:
#     context_text = f.read()

PARSE_CONTEXT_TURNS = False  # True if your paste uses User:/Assistant: markers

if context_text.strip():
    print(f"Context loaded ({len(context_text)} characters).")
else:
    print("No context — pricing this prompt as a standalone, first-turn request.")

## 5. Optional: system prompt
System prompt tokens count too — include one here if your real request will use one. Leave blank if not.

In [ ]:
system_prompt = ""  # leave as empty string if you don't use one

## 6. Count tokens (exact, via the API — free, no generation happens)

Counts the request as it would actually be sent: system prompt + context + new prompt.

The context and prompt are counted separately so section 7 can show you how much of
your bill is the new prompt and how much is re-sending what you already sent. The
prompt figure is a *marginal* count — total minus context — which is the number that
actually matters and sidesteps per-message framing overhead.

In [ ]:
est = tce.estimate(
    client,
    model,
    prompt_text=prompt_text,
    context_text=context_text,
    system_prompt=system_prompt,
    parse_turns=PARSE_CONTEXT_TURNS,
)

tce.print_counts(est)

## 7. Estimate cost

Input cost is exact (based on the real counts above). Output cost is a guess — enter roughly how many tokens you expect Claude to generate in response (a rough rule of thumb is ~0.75 tokens per word for plain English prose; more for code).

Three views, all in dollars:

**(a) This turn** — split into what you're paying to re-send versus what you're paying for the new prompt.

**(b) The next N turns** — because context is resent every request, a conversation's cost grows *quadratically*, not linearly. Turn 20 pays for turns 1–19 all over again.

**(c) With prompt caching** — the same N turns if you cache the stable prefix. Cache reads cost 0.1× and writes 1.25×, so on a long conversation with a big context this is usually the difference between a rounding error and a real bill.

In [ ]:
estimated_output_tokens = 500  # <-- change this to whatever you expect the response length to be
turns_to_project = 10          # <-- how many more turns to project in views (b) and (c)
cache_ttl = "5m"               # "5m" or "1h" — the cache TTL priced in view (c)

projection = tce.project(est, estimated_output_tokens, turns_to_project)
cached = tce.project_cached(
    est,
    estimated_output_tokens,
    turns_to_project,
    ttl=cache_ttl,
    uncached_total=projection.uncached_total,
)

tce.print_cost(est, projection, cached)

## 8. (Optional) Batch-check multiple prompts at once

Useful if you want to estimate cost across several prompts before running a batch (e.g. many podcast script drafts, citation checks, etc.).

Each prompt is priced against the same `context_text` from section 4, so this also answers "I have one big context and several candidate prompts — what does each cost?" Set `APPLY_CONTEXT_TO_BATCH = False` to price them standalone instead.

In [ ]:
prompts = [
    "First prompt text goes here.",
    "Second prompt text goes here.",
    # add as many as you want
]

assumed_output_tokens_each = 500
APPLY_CONTEXT_TO_BATCH = True  # False to price each prompt with no context

tce.print_batch(
    tce.batch(
        client,
        model,
        prompts,
        context_text=context_text,
        system_prompt=system_prompt,
        assumed_output_tokens=assumed_output_tokens_each,
        apply_context=APPLY_CONTEXT_TO_BATCH,
        parse_turns=PARSE_CONTEXT_TURNS,
    )
)